This exercise will do the following:
Allow the user to search airline ticket prices, the system will search the internet and provide the list of top 3 options to the user.
If user confirms then the system will save the user selection and information in SQLlite DB (it will not actually do the booking itself).
The user can also fetch the information saved.

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr




In [2]:


# As an alternative, if you'd like to use Ollama instead of OpenAI
# Check that Ollama is running for you locally (see week1/day2 exercise) then uncomment these next 2 lines
# MODEL = "llama3.2"
# openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI()

OpenAI API Key exists and begins sk-proj-


In [3]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [38]:
# set up the database for ticekt prices

import sqlite3

DB = "flight_prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS flight_prices (city TEXT PRIMARY KEY,price REAL,date TEXT)
    """)
    conn.commit()

def set_ticket_price(city, price):
    print(f"DATABASE TOOL CALLED: set price for {city}", flush=True)

    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO flight_prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()
        return f"Ticket price to {city} set to ${price}"

def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM flight_prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"




In [ ]:
#with sqlite3.connect(DB) as conn:
#    cursor = conn.cursor()
#    cursor.execute("""
#    DROP TABLE flight_prices  
#    """)
#    conn.commit()

In [39]:
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

func_map = {
    "set_ticket_price": set_ticket_price,
    "get_ticket_price": get_ticket_price
}        

DATABASE TOOL CALLED: set price for london
DATABASE TOOL CALLED: set price for paris
DATABASE TOOL CALLED: set price for tokyo
DATABASE TOOL CALLED: set price for sydney


In [40]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["city"],
        "additionalProperties": False
    }
}

price_function_set = {
	"name": "set_ticket_price",
	"description": "Set the price of a return ticket to the destination city.",
	"parameters": {
		"type": "object",
		"properties": {
			"city": {
				"type": "string",
				"description": "The city that the customer wants to travel to"
			},
			"price": {
				"type": "number",
				"description": "Set the price for city travel"
			}
		},
		"required": [
			"city","price"
		],
		"additionalProperties": False
	}
}
tools = [{"type": "function", "function": price_function}, {"type": "function", "function": price_function_set}]

In [41]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        func = func_map[tool_call.function.name]
        func_args = json.loads(tool_call.function.arguments)
        func_result = func(**func_args)
        print(f"handle_tool_calls Called : {tool_call.function.name},  {func_args}",{func_result}, flush=True)

        responses.append({
            "role": "tool",
            "content": func_result,
            "tool_call_id": tool_call.id
        })
    return responses


def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    print(f"TOOL Called MEssage 1 {messages}", flush=True)
 
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        print(f"TOOL Called  {response.choices[0].message}", flush=True)

        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        print(f"TOOL Called MEssage 2 {messages}", flush=True)

        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

    

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7873
* To create a public link, set `share=True` in `launch()`.


TOOL Called MEssage 1 [{'role': 'system', 'content': "\nYou are a helpful assistant for an Airline called FlightAI.\nGive short, courteous answers, no more than 1 sentence.\nAlways be accurate. If you don't know the answer, say so.\n"}, {'role': 'user', 'content': 'please set ticket price to Rome to 700'}]
TOOL Called  ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_mWJx2YZFcYerKO0ug9QDQ5cm', function=Function(arguments='{"city":"Rome","price":700}', name='set_ticket_price'), type='function')])
DATABASE TOOL CALLED: set price for Rome
handle_tool_calls Called : set_ticket_price,  {'city': 'Rome', 'price': 700} {'Ticket price to Rome set to $700'}
TOOL Called MEssage 2 [{'role': 'system', 'content': "\nYou are a helpful assistant for an Airline called FlightAI.\nGive short, courteous answers, no more than 1 sentence.\nAlways be accurate. If you don't know the an